In [ ]:
### ============================================================
### Lecture 1 — Model Assessment
###               and Forecast Evaluation Metrics
### Data: Illustrative stock-market data (same seed as Lecture 1.3)
### ============================================================

> **Where we left off:**
> In Lectures 1.3–1.5 we built four regressions on stock market data.
> Some looked impressive in-sample, some looked weak.
>
> **Today:** We put all four through a proper train/test evaluation.
> A model that looks good in-sample does not necessarily forecast well —
> and the metrics we compute will show exactly *why* and *how much* each one fails.

---
## Setup — Rebuild the Data & Fit All Four Models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from sklearn.linear_model import LinearRegression
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)

C_BLUE   = '#2166AC'
C_ORANGE = '#D6604D'
C_GREEN  = '#1A9850'
C_RED    = '#D73027'
C_GRAY   = '#888888'
C_TRAIN  = '#DDEEFF'   # light blue — training window
C_TEST   = '#FFEEDD'   # light orange — test window

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#F9F9F9',
    'axes.edgecolor': '#CCCCCC', 'axes.grid': True,
    'grid.color': '#EEEEEE', 'grid.linewidth': 0.8,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 8,  'ytick.labelsize': 8,
})

In [ ]:
# ── data (identical construction to Lecture 1.3) ─────────────────────
dates = pd.bdate_range('2020-01-02', '2024-12-31')
N     = len(dates)

sp_ret          = np.random.normal(0.0005, 0.011, N)
sp_ret[25:55]  -= 0.025
sp_ret[55:90]  += 0.020
sp_ret[500:620]-= 0.005
sp_ret          = np.clip(sp_ret, -0.12, 0.10)
sp_price        = 3300 * np.exp(np.cumsum(sp_ret))

vix = np.clip(18 + np.random.normal(0, 3, N) - 200 * sp_ret, 10, 85)
vix[25:55] += 30; vix[55:90] -= 10; vix[500:620] += 10
vix = np.clip(vix, 10, 85)

aapl_ret = 0.6 * sp_ret + np.random.normal(0, 0.008, N)
msft_ret = 0.7 * sp_ret + np.random.normal(0, 0.007, N)

df = pd.DataFrame({
    'sp_ret': sp_ret, 'sp_price': sp_price,
    'vix': vix, 'aapl_ret': aapl_ret, 'msft_ret': msft_ret
}, index=dates)

# ── 80/20 chronological split ─────────────────────────────────────────
SPLIT      = int(N * 0.8)
split_date = dates[SPLIT]
train_idx  = dates[:SPLIT]
test_idx   = dates[SPLIT:]

print(f'Total : {N} trading days')
print(f'Train : {SPLIT} days  ({train_idx[0].date()} → {train_idx[-1].date()})')
print(f'Test  : {N-SPLIT} days  ({test_idx[0].date()} → {test_idx[-1].date()})')

In [ ]:
# ── fit all four models on training data only ─────────────────────────

# M1: MSFT ~ AAPL  (cross-sectional, genuine signal)
X1_tr = df.loc[train_idx, 'aapl_ret'].values.reshape(-1, 1)
y1_tr = df.loc[train_idx, 'msft_ret'].values
X1_te = df.loc[test_idx,  'aapl_ret'].values.reshape(-1, 1)
y1_te = df.loc[test_idx,  'msft_ret'].values
reg1  = LinearRegression().fit(X1_tr, y1_tr)

# M2: SP_ret ~ VIX  (time-indexed, genuine signal)
X2_tr = df.loc[train_idx, 'vix'].values.reshape(-1, 1)
y2_tr = df.loc[train_idx, 'sp_ret'].values
X2_te = df.loc[test_idx,  'vix'].values.reshape(-1, 1)
y2_te = df.loc[test_idx,  'sp_ret'].values
reg2  = LinearRegression().fit(X2_tr, y2_tr)

# M3: AR(1) on returns  (weak signal — near-zero R²)
lag_ret = pd.DataFrame({'yt': df['sp_ret'],
                        'yt_1': df['sp_ret'].shift(1)}).dropna()
tr3 = lag_ret.index < split_date
te3 = lag_ret.index >= split_date
X3_tr = lag_ret.loc[tr3, 'yt_1'].values.reshape(-1, 1)
y3_tr = lag_ret.loc[tr3, 'yt'].values
X3_te = lag_ret.loc[te3, 'yt_1'].values.reshape(-1, 1)
y3_te = lag_ret.loc[te3, 'yt'].values
reg3  = LinearRegression().fit(X3_tr, y3_tr)

# M4: AR(2) on price level  (spuriously high R²)
lag_px = pd.DataFrame({'yt':   df['sp_price'],
                       'yt_1': df['sp_price'].shift(1),
                       'yt_2': df['sp_price'].shift(2)}).dropna()
tr4 = lag_px.index < split_date
te4 = lag_px.index >= split_date
X4_tr = lag_px.loc[tr4, ['yt_1','yt_2']].values
y4_tr = lag_px.loc[tr4, 'yt'].values
X4_te = lag_px.loc[te4, ['yt_1','yt_2']].values
y4_te = lag_px.loc[te4, 'yt'].values
reg4  = LinearRegression().fit(X4_tr, y4_tr)

print('All four models fitted on training data only.')

---
## Section 1 — In-Sample vs Out-of-Sample: The Core Distinction

```
Timeline ────────────────────────────────────────────────────────────────▶

│◄──────────── Training set (80%) ────────────────────►│◄── Test set (20%) ──►│
│  Jan 2020                                Jan 2024    │  Jan 2024   Dec 2024 │
│                                                       │                      │
│  Model sees this data → learns β                      │  Model never saw     │
│  In-sample fit = how well β describes the past        │  Out-of-sample fit = │
│                                                       │  honest forecast     │
│                                                       │  quality             │
```

**The golden rule:** A model is only as good as its out-of-sample performance.
In-sample R² tells you how well the model memorised the past.
Out-of-sample R² tells you whether that memory is actually useful for the future.

| | In-sample | Out-of-sample |
|---|---|---|
| Data used to fit | ✓ yes | ✗ no — model never saw it |
| What it measures | Fit to the past | Forecast quality |
| Risk | Overfitting inflates it | Honest but can be negative |

> **Out-of-sample R² < 0** is possible and meaningful.
> It means the model is literally worse than just predicting the mean every day.
> We will see this happen with Model 3.

---
## Section 2 — Visual Comparison: Fitted vs Forecast for All Four Models

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(15, 20), sharex=False)
fig.suptitle(
    'In-Sample Fit vs Out-of-Sample Forecast  |  Four Models\n'
    'Blue region = training window  |  Orange region = test window',
    fontsize=13, fontweight='bold', y=1.01)

configs = [
    # (title, idx_tr, y_tr, fit_tr, idx_te, y_te, fit_te, y_label)
    ('M1  MSFT ~ AAPL  |  cross-sectional, genuine correlation',
     train_idx, y1_tr * 100, reg1.predict(X1_tr) * 100,
     test_idx,  y1_te * 100, reg1.predict(X1_te) * 100,
     'Daily return (%)'),
    ('M2  S&P 500 return ~ VIX  |  fear index, genuine signal',
     train_idx, y2_tr * 100, reg2.predict(X2_tr) * 100,
     test_idx,  y2_te * 100, reg2.predict(X2_te) * 100,
     'Daily return (%)'),
    ('M3  AR(1) on returns  |  weak signal, overfits noise',
     lag_ret.index[tr3], y3_tr * 100, reg3.predict(X3_tr) * 100,
     lag_ret.index[te3], y3_te * 100, reg3.predict(X3_te) * 100,
     'Daily return (%)'),
    ('M4  AR(2) on price level  |  spuriously high in-sample R²',
     lag_px.index[tr4], y4_tr, reg4.predict(X4_tr),
     lag_px.index[te4], y4_te, reg4.predict(X4_te),
     'Price level (USD)'),
]

for ax, (title, itr, ytr, ftr, ite, yte, fte, ylabel) in zip(axes, configs):
    r2_in  = 1 - np.sum((ytr-ftr)**2) / np.sum((ytr-ytr.mean())**2)
    r2_out = 1 - np.sum((yte-fte)**2) / np.sum((yte-yte.mean())**2)

    ax.axvspan(itr[0], itr[-1], color=C_TRAIN, alpha=0.6, zorder=0)
    ax.axvspan(ite[0], ite[-1], color=C_TEST,  alpha=0.6, zorder=0)

    ax.plot(itr, ytr, color=C_BLUE,   lw=0.7, alpha=0.55, label='Actual (train)')
    ax.plot(ite, yte, color=C_ORANGE, lw=0.7, alpha=0.55, label='Actual (test)')
    ax.plot(itr, ftr, color=C_BLUE,   lw=1.8, ls='--', label=f'Fitted  (in-sample  R²={r2_in:.4f})')
    ax.plot(ite, fte, color=C_RED,    lw=2.0, ls='--', label=f'Forecast (out-of-sample R²={r2_out:.4f})')

    if ylabel == 'Price level (USD)':
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8, loc='upper left', ncol=2)

    # vertical line at split
    ax.axvline(split_date, color=C_GRAY, lw=1.2, ls=':')
    ax.text(split_date, ax.get_ylim()[1],
            '  train|test', fontsize=7.5, color=C_GRAY, va='top')

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

---
## Section 3 — Forecast Evaluation Metrics

Define forecast error at each time step:

&emsp; **e_t = y_t − ŷ_t** &emsp; (actual minus forecast)

---

### MAE — Mean Absolute Error

&emsp; **MAE = mean( |e_t| )**

Take each error, ignore the sign, average them.
What it says in plain English: *on a typical day, my forecast is off by X units.*
Every error counts equally — a miss of 2% is twice as bad as a miss of 1%.

---

### RMSE — Root Mean Squared Error

&emsp; **RMSE = √mean( e_t² )**

Square each error before averaging, then take the square root.
What it says: *my typical error size, but large errors are punished extra.*
Because of squaring, a miss of 4% hurts 4× more than a miss of 2%.
RMSE > MAE always. The gap between them tells you how much the big outlier errors dominate.

---

### MAPE — Mean Absolute Percentage Error

&emsp; **MAPE = mean( |e_t / y_t| ) × 100%**

Normalise each error by the actual value, so the scale of y does not matter.
What it says: *on average, I'm off by X% of the true value.*
Useful for comparing models across different series.

⚠️ **One important limitation:** if y_t is close to zero (e.g. daily returns
near zero), dividing by y_t blows up. MAPE is unreliable for returns data.
We include it here for completeness but will rely on MAE and RMSE for returns.

---

### R² — Coefficient of Determination

&emsp; **R² = 1 − SS_res / SS_tot** &emsp; where &emsp; SS_res = Σ e_t² &emsp; and &emsp; SS_tot = Σ (y_t − ȳ)²

What it says: *what fraction of the total variation in y does my model explain?*
R² = 1 → perfect fit. R² = 0 → no better than predicting the mean.
R² < 0 → worse than predicting the mean. This can happen out-of-sample.

**Out-of-sample R² is the single most important number here.**
It directly answers: does this model add any value beyond the trivial benchmark?

---
## Section 4 — Metrics for All Four Models

In [ ]:
def compute_metrics(y_true, y_pred):
    e     = y_true - y_pred
    mae   = np.mean(np.abs(e))
    rmse  = np.sqrt(np.mean(e ** 2))
    ss_res = np.sum(e ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2    = 1 - ss_res / ss_tot
    with np.errstate(divide='ignore', invalid='ignore'):
        pct = np.abs(e / np.where(np.abs(y_true) > 1e-8, y_true, np.nan))
        mape = np.nanmean(pct) * 100
    return dict(mae=mae, rmse=rmse, mape=mape, r2=r2)

# returns scale → multiply by 100 to express as %
# price level   → keep raw (USD)
model_specs = [
    ('M1  MSFT ~ AAPL',        X1_tr, y1_tr, X1_te, y1_te, reg1, 100),
    ('M2  SP_ret ~ VIX',       X2_tr, y2_tr, X2_te, y2_te, reg2, 100),
    ('M3  AR(1) on returns',   X3_tr, y3_tr, X3_te, y3_te, reg3, 100),
    ('M4  AR(2) on price lvl', X4_tr, y4_tr, X4_te, y4_te, reg4, 1),
]

results = []
for name, Xtr, ytr, Xte, yte, reg, sc in model_specs:
    m_in  = compute_metrics(ytr * sc, reg.predict(Xtr) * sc)
    m_out = compute_metrics(yte * sc, reg.predict(Xte) * sc)
    results.append((name, m_in, m_out, sc))

# ── print table ──────────────────────────────────────────────────────
hdr = f'{"":<26}  {"MAE":>9}  {"RMSE":>9}  {"MAPE":>9}  {"R²":>8}'
print(hdr)
print('─' * 70)
for name, m_in, m_out, sc in results:
    unit = '%' if sc == 100 else 'USD'
    print(f'{name} [{unit}]')
    print(f'  {"In-sample":24}  {m_in["mae"]:9.5f}  {m_in["rmse"]:9.5f}  '
          f'{m_in["mape"]:9.1f}%  {m_in["r2"]:8.4f}')
    print(f'  {"Out-of-sample":24}  {m_out["mae"]:9.5f}  {m_out["rmse"]:9.5f}  '
          f'{m_out["mape"]:9.1f}%  {m_out["r2"]:8.4f}')
    print()

---
## Section 5 — Metrics Dashboard

The table above is dense. The chart below makes the key comparisons visual:
- **Top row**: MAE and RMSE side by side (in vs out), for the three returns models
- **Bottom left**: R² in vs out — the overfit fingerprint
- **Bottom right**: R² degradation (in minus out) — how much each model loses when forecasting

In [ ]:
# ── only M1/M2/M3 for returns-scale metrics (M4 has different units) ─
names3   = [r[0] for r in results[:3]]
mae_in3  = [r[1]['mae']  for r in results[:3]]
mae_out3 = [r[2]['mae']  for r in results[:3]]
rmse_in3 = [r[1]['rmse'] for r in results[:3]]
rmse_out3= [r[2]['rmse'] for r in results[:3]]
r2_in4   = [r[1]['r2']   for r in results]
r2_out4  = [r[2]['r2']   for r in results]
names4   = [r[0] for r in results]
degradation = [i - o for i, o in zip(r2_in4, r2_out4)]

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('Forecast Evaluation Metrics — Four Models',
             fontsize=14, fontweight='bold')

x3 = np.arange(3)
x4 = np.arange(4)
w  = 0.38

# ── top-left: MAE ────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 0])
b1 = ax.bar(x3 - w/2, mae_in3,  w, color=C_BLUE,   alpha=0.85, label='In-sample')
b2 = ax.bar(x3 + w/2, mae_out3, w, color=C_ORANGE, alpha=0.85, label='Out-of-sample')
ax.set_xticks(x3)
ax.set_xticklabels([n.split()[0]+' '+n.split()[1] for n in names3], fontsize=9)
ax.set_ylabel('MAE  (% daily return)')
ax.set_title('MAE  —  Mean Absolute Error\n'
             'average |error| per day', fontweight='bold')
ax.legend(fontsize=9)
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.002,
            f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=7.5)

# ── top-right: RMSE ───────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 1])
b3 = ax.bar(x3 - w/2, rmse_in3,  w, color=C_BLUE,   alpha=0.85, label='In-sample')
b4 = ax.bar(x3 + w/2, rmse_out3, w, color=C_ORANGE, alpha=0.85, label='Out-of-sample')
ax.set_xticks(x3)
ax.set_xticklabels([n.split()[0]+' '+n.split()[1] for n in names3], fontsize=9)
ax.set_ylabel('RMSE  (% daily return)')
ax.set_title('RMSE  —  Root Mean Squared Error\n'
             'large errors penalised more heavily', fontweight='bold')
ax.legend(fontsize=9)
for b in list(b3) + list(b4):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.002,
            f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=7.5)

# ── bottom-left: R² in vs out scatter ─────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
colors4 = [C_GREEN, C_GREEN, C_ORANGE, C_RED]
lo = min(r2_in4 + r2_out4) - 0.08
hi = max(r2_in4 + r2_out4) + 0.08
ax.plot([lo, hi], [lo, hi], color=C_GRAY, lw=1.2, ls='--',
        label='In = Out (perfect generalisation)', zorder=1)
ax.axhline(0, color=C_RED, lw=0.9, ls=':', alpha=0.7, label='R² = 0 (mean benchmark)')
short_names = ['M1 MSFT~AAPL', 'M2 SP~VIX', 'M3 AR(1)', 'M4 Price lvl']
for i, (sn, r2i, r2o, col) in enumerate(zip(short_names, r2_in4, r2_out4, colors4)):
    ax.scatter(r2i, r2o, color=col, s=140, zorder=4,
               edgecolors='white', lw=1.2)
    ax.annotate(sn, (r2i, r2o), xytext=(8, 5),
                textcoords='offset points', fontsize=8.5,
                color=col, fontweight='bold')
ax.set_xlabel('In-sample R²')
ax.set_ylabel('Out-of-sample R²')
ax.set_title('R²  —  In vs Out\nPoints below diagonal = performance degraded',
             fontweight='bold')
ax.legend(fontsize=8.5)

# shade below diagonal = overfitting zone
ax.fill_between([lo, hi], [lo, hi], lo,
                alpha=0.06, color=C_RED, label='_nolegend_')
ax.text((lo+hi)/2, lo + 0.04, 'overfitting zone',
        color=C_RED, fontsize=8, alpha=0.7, ha='center')

# ── bottom-right: R² degradation bar ─────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
bar_colors = [C_GREEN if d < 0.15 else C_ORANGE if d < 0.3 else C_RED
              for d in degradation]
bars = ax.bar(x4, degradation, color=bar_colors, alpha=0.85, width=0.5)
ax.set_xticks(x4)
ax.set_xticklabels(short_names, fontsize=8.5)
ax.set_ylabel('R² degradation  (in − out)')
ax.set_title('R² Degradation\n'
             'how much performance drops out-of-sample', fontweight='bold')
ax.axhline(0, color=C_GRAY, lw=0.8)
for bar, dval in zip(bars, degradation):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.003,
            f'{dval:.4f}', ha='center', va='bottom', fontsize=8,
            fontweight='bold')

plt.show()

---
## Section 6 — Who Wins, Who Fails, and Why

### The verdict

| Model | In-sample R² | Out-of-sample R² | Verdict |
|-------|-------------|-----------------|--------|
| M2  SP_ret ~ VIX | 0.41 | 0.30 | ✅ Best overall — real signal, generalises |
| M1  MSFT ~ AAPL | 0.30 | 0.20 | ✅ Good — structural correlation holds |
| M3  AR(1) returns | 0.06 | **−0.11** | ❌ Worst — overfits noise, worse than mean |
| M4  AR(2) price level | 0.997 | 0.987 | ⚠️ Misleading — high R² is spurious |

---

### M2 and M1 — why they hold up

Both models capture a **structural economic relationship** that exists in the real world:
- VIX is a direct measure of market fear — when VIX rises, investors sell, returns fall.
  This relationship is not a statistical accident; it is a market mechanism.
- AAPL and MSFT both move with the overall tech sector.
  The correlation reflects common exposure to the same macro factors.

A genuine relationship in the training data tends to persist into the test data.
The R² drops (from 0.41 to 0.30 for M2) because the test period has its own
idiosyncratic noise — but the signal survives.

---

### M3 — why it goes negative

AR(1) on returns had in-sample R² = 0.06 — already tiny, but still positive.
Out-of-sample it drops to **−0.11**.

What happened? In-sample, the model found a very small pattern in the residuals
that turned out to be **noise, not signal**. It fit to randomness.
When that randomness did not repeat in the test set, the "pattern" actively hurt.

R² < 0 has a clean interpretation:
> ŷ_t = ȳ (just predicting the mean every day) would have been better.
> The model's "learned" coefficient made things worse, not better.

This is the definition of overfitting: finding structure that does not generalise.

---

### M4 — why the high R² is misleading

AR(2) on price level has R² = 0.997 in-sample. Sounds incredible.
But the model has learned exactly one thing: **tomorrow's price ≈ today's price**.
On a trending series, that is trivially true and has nothing to do with forecasting ability.

The out-of-sample RMSE jumps from 37 USD to 58 USD — a 56% increase.
The model is not capturing any real return-generating process;
it is just following the trend it was shown in training.

This is why we transform to log returns before modelling: it removes this illusion.

In [ ]:
print('=' * 68)
print('LECTURE 1.7 — SUMMARY')
print('=' * 68)
print()
print('The four metrics, in plain English:')
print()
print('  MAE  = mean|e_t|              On a typical day, I am off by X units.')
print('  RMSE = sqrt(mean(e_t^2))      Same, but large errors hurt much more.')
print('  MAPE = mean|e_t/y_t|*100      How big is my error relative to the true value?')
print('         WARNING: blows up when y_t is close to zero (daily returns).')
print('  R²   = 1 - SS_res/SS_tot      What fraction of variation did I explain?')
print('         R² < 0 out-of-sample = model is worse than predicting the mean.')
print()
print('─' * 68)
print()
print('Model rankings (out-of-sample R², highest to lowest):')
print()
ranked = sorted(zip(short_names, r2_out4), key=lambda x: -x[1])
for rank, (nm, r2o) in enumerate(ranked, 1):
    r2i = r2_in4[short_names.index(nm)]
    gap = r2i - r2o
    flag = '  <-- NEGATIVE: worse than mean' if r2o < 0 else ''
    flag = '  <-- spurious (non-stationary)' if nm == 'M4 Price lvl' else flag
    print(f'  #{rank}  {nm:<18}  out-R²={r2o:+.4f}  drop={gap:.4f}{flag}')
print()
print('Key lesson:')
print('  High in-sample R² is not evidence of forecast quality.')
print('  Only out-of-sample evaluation can tell you whether a model actually works.')